In [27]:
import pandas as pd
import numpy as np


In [17]:
# Reproducibility
np.random.seed(42)
n_days = 1000

In [18]:
print("Generating highly realistic dataset based on SL Current Data...")
# 1. Base values
base_usd = 313.25
base_diesel = 382.0
base_petrol = 398.0
base_rice = 230.0 # නාඩු
base_bread = 130.0 # පාන්
base_lorry_quota = 200
base_bike_quota = 8


Generating highly realistic dataset based on SL Current Data...


In [19]:
# 2. USD Rate (ඩොලර් අනුපාතය) - රු. 300 ත් 380 ත් අතර වෙනස් වේ යැයි උපකල්පනය කරමු
usd_rate = base_usd + np.random.normal(0, 15, n_days) # සාමාන්‍යයෙන් රු. 313.25 වන අතර විචලනය ±15
usd_rate = np.clip(usd_rate, 300, 380) # 300 සහ 380 අතර සීමා කරන්න

In [20]:
# 3. Fuel Prices (ඉන්ධන මිල - ඩොලර් එක මත තදින් රඳා පවතී)
diesel_price = (usd_rate * 1.15) + np.random.normal(loc=20, scale=10, size=n_days) # ඩොලර් එක මත රු. 1.15 ගුණ කිරීම
diesel_price = np.clip(diesel_price, 300, 500)

petrol_price = diesel_price + 16 + np.random.normal(loc=0, scale=5, size=n_days) # පෙට්‍රල් මිල ඩීසල් වලට වඩා තරමක් වැඩියි

In [21]:
# 4. QR Quotas (කෝටාව - ඉන්ධන මිල 400 පැන්නොත් කෝටාව අඩු කරයි)
quota_multiplier = np.where(diesel_price > 420, 0.5, # දරුණු අර්බුදයකදී 50% කින් කපයි
                   np.where(diesel_price > 380, 0.8, 1.0))

lorry_quota = np.round(base_lorry_quota * quota_multiplier + np.random.normal(0, 5, n_days)) # කෝටාව ±5 කින් වෙනස් වේ
bike_quota = np.round(base_bike_quota * quota_multiplier + np.random.normal(0, 1, n_days)) # කෝටාව ±1 කින් වෙනස් වේ

In [22]:
# 5. Food Prices (ආහාර මිල - ප්‍රවාහන ගාස්තු (ඩීසල්) සහ ආනයන (ඩොලර්) මත රඳා පවතී)
rice_price = base_rice + (diesel_price - base_diesel) * 0.35 + (usd_rate - base_usd) * 0.2 + np.random.normal(0, 5, n_days) # ඩීසල් මිල සහ ඩොලර් අනුපාතය මත රඳා පවතී
bread_price = base_bread + (usd_rate - base_usd) * 0.15 + (diesel_price - base_diesel) * 0.1 + np.random.normal(0, 3, n_days) # ඩොලර් අනුපාතය සහ ඩීසල් මිල මත රඳා පවතී


In [23]:
# 6. Power Cuts (විදුලි කප්පාදුව - ඩීසල් නැති වීම සහ ඩොලර් හිඟය නිසා)
# දැනට කප්පාදුවක් නෑ (0), නමුත් අවදානම වැඩි වෙද්දි පැය 1-4 අතර කප්පාදුවක් එන්න පුළුවන්
power_cut_prob = ((diesel_price - 380) / 100) + ((usd_rate - 310) / 100)
power_cut_hours = np.where(power_cut_prob > 0.4, np.random.randint(1, 5, n_days), 0)
power_cut_hours = np.clip(power_cut_hours, 0, 8)

In [24]:
# 7. Create DataFrame
df = pd.DataFrame({
    'USD_Rate': np.round(usd_rate, 2),
    'Diesel_Price_Rs': np.round(diesel_price, 2),
    'Petrol_Price_Rs': np.round(petrol_price, 2),
    'Lorry_Quota_L': np.clip(lorry_quota, 50, 200),
    'Bike_Quota_L': np.clip(bike_quota, 2, 8),
    'Rice_Price_Rs': np.round(np.clip(rice_price, 180, 350), 2),
    'Bread_Price_Rs': np.round(np.clip(bread_price, 100, 250), 2),
    'Power_Cut_Hours': power_cut_hours
})


In [25]:
# ==========================================
# TARGET VARIABLES (අපේ ML Models වලට අනාවැකි කියන්න)
# ==========================================

# Target 1: Economic Status (Multi-class Classification සඳහා: Softmax)
conditions = [
    (df['Power_Cut_Hours'] > 0) & (df['Rice_Price_Rs'] > 250), # විදුලියත් නෑ, කෑමත් ගණන්
    (df['Diesel_Price_Rs'] > 400) | (df['USD_Rate'] > 330)     # තෙල් ගණන්, ඩොලරය ඉහළයි
]
choices = ['Severe_Crisis', 'Moderate_Stress']
df['Economic_Status'] = np.select(conditions, choices, default='Normal')

# Target 2: Power Cut Warning (Binary Classification සඳහා: Logistic Regression / SVM)
df['Power_Cut_Warning'] = np.where(df['Power_Cut_Hours'] > 0, 1, 0)

In [26]:
# CSV එක Save කරගමු
df.to_csv('../data/sl_economic_crisis_data.csv', index=False)
print("✅ Dataset generated and saved as 'sl_economic_crisis_data.csv'!")


✅ Dataset generated and saved as 'sl_economic_crisis_data.csv'!
